# CrisisOps v2 — GRPO Training Notebook

Self-contained Colab notebook for training the CrisisOps PM agent using GRPO
with Qwen2.5-1.5B-Instruct and Unsloth LoRA.

**Run order:**
1. Install dependencies
2. Clone repo and run calibration
3. Run GRPO training
4. Plot reward and cross-verification curves

In [ ]:
# Cell 1: Install dependencies
# Run this first and restart the runtime after installation.
!pip install -q unsloth trl>=0.29.0 transformers accelerate peft
!pip install -q fastapi uvicorn pydantic requests pytest

In [ ]:
# Cell 2: Set up the CrisisOps package path
import sys
import os

# If running from a cloned repo, add the root to sys.path
REPO_ROOT = '/content/crisisops'  # adjust if needed
if os.path.exists(REPO_ROOT):
    sys.path.insert(0, REPO_ROOT)
else:
    # Fallback: assume we're running from the repo root
    sys.path.insert(0, '.')

print('Python path:', sys.path[:3])

In [ ]:
# Cell 3: Run calibration BEFORE training
# This verifies the greedy PM scores 0.45-0.55 and oracle scores 0.70-0.80.
# If calibration fails, adjust inflation_bias in env/candor.py.
from calibration.calibrate import run_calibration

run_calibration()

In [ ]:
# Cell 4: GRPO Training
# Trains Qwen2.5-1.5B-Instruct with LoRA r=16 using counterfactual reward.
# Curriculum starts at Level 1 and unlocks higher levels automatically.

from training.grpo_trainer import train

train(
    curriculum_level=1,
    num_episodes=200,       # reduce for quick test; use 500+ for full training
    output_dir='./outputs/crisisops_grpo',
    seed=42,
)

In [ ]:
# Cell 5: Evaluate trained agent vs greedy baseline
import random
from env.environment import CrisisOpsEnv, MAX_STEPS
from reward.baseline import GreedyPMBaseline
from reward.counterfactual import project_score
from scenarios.level1 import get_random_level1_scenario

N_EVAL = 10
greedy_scores = []

for i in range(N_EVAL):
    env = CrisisOpsEnv(scenario_fn=get_random_level1_scenario(), curriculum_level=1)
    env.reset(seed=2000 + i)
    baseline = GreedyPMBaseline()
    done = False
    step = 0
    while not done and step < MAX_STEPS:
        action = baseline.act(env._state)
        _, _, done, _ = env.step(action)
        step += 1
    greedy_scores.append(project_score(env._state))

print(f'Greedy PM mean score: {sum(greedy_scores)/len(greedy_scores):.3f}')
print('(Target: 0.45–0.55)')

In [ ]:
# Cell 6: Plot training curves
# Reads training logs from the output directory.
import json
import os

try:
    import matplotlib.pyplot as plt
    import matplotlib
    matplotlib.use('Agg')
except ImportError:
    print('matplotlib not installed; install with: pip install matplotlib')
    raise

log_path = './outputs/crisisops_grpo/trainer_state.json'
if not os.path.exists(log_path):
    print(f'Log file not found at {log_path}. Run training first.')
else:
    with open(log_path) as f:
        logs = json.load(f)

    rewards = [entry.get('train/reward', 0) for entry in logs.get('log_history', [])]
    steps   = list(range(len(rewards)))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(steps, rewards)
    ax1.axhline(0.15, color='orange', linestyle='--', label='Level 2 threshold')
    ax1.axhline(0.25, color='red',    linestyle='--', label='Level 3 threshold')
    ax1.axhline(0.35, color='purple', linestyle='--', label='Level 4 threshold')
    ax1.set_xlabel('Training step')
    ax1.set_ylabel('Counterfactual reward')
    ax1.set_title('Training Reward Curve')
    ax1.legend()

    # Cross-verification rate if logged
    cvr = [entry.get('train/cross_verify_rate', 0) for entry in logs.get('log_history', [])]
    ax2.plot(steps, cvr, color='green')
    ax2.set_xlabel('Training step')
    ax2.set_ylabel('Cross-verification rate')
    ax2.set_title('Cross-Verification Rate')

    plt.tight_layout()
    plt.savefig('./outputs/training_curves.png', dpi=150)
    print('Saved training_curves.png')
    plt.show()

In [ ]:
# Cell 7: Inspect CrisisGenerator difficulty report
from env.crisis_generator import CrisisGenerator
import pprint

gen = CrisisGenerator(curriculum_level=1)
# Simulate some episodes with varied outcomes
import random
rng = random.Random(42)

from env.environment import CrisisOpsEnv, MAX_STEPS
from reward.baseline import GreedyPMBaseline
from scenarios.level1 import LEVEL1_SCENARIOS

for i in range(5):
    fn = rng.choice(LEVEL1_SCENARIOS)
    env = CrisisOpsEnv(scenario_fn=fn, curriculum_level=1)
    env.reset(seed=i)
    baseline = GreedyPMBaseline()
    done = False
    step = 0
    while not done and step < MAX_STEPS:
        action = baseline.act(env._state)
        _, _, done, _ = env.step(action)
        step += 1
    gen.update_after_episode(env._state, env._state.recovery_pct())

pprint.pprint(gen.get_difficulty_report())